# What is an SDK?

SDK stands for software development kit. It is a package of code/tools that can help developers use an API more easily. It includes all the boilerplate code, data models, and even error handling. Instead of having to have developers rely on your API documentation, you can provide them with (or make your own) SDK so that the code is more easily readable / callable.

lets define out all the endpoints we have access to based on the inspect element calls
- current
- 1day
- 2day
- 3day
- 7day
- 12hour
- fifteenminute

we can create a class that has methods for each endpoint. Since the requests all have the same parameters, we also wont need to redefine them each time either

```python
class WeatherAPI:
    def __init__(self, 
                 latitude: str, 
                 longitude: str,
                 units: str = "e",
                 language: str = "en-US",
                 format: str = "json",
                 apiKey: str = "71f92ea9dd2f4790b92ea9dd2f779061"
                 ):
        
        # we know that each request is going to require these parameters. the latitude and longitude may change with each request, so they do not have default values and are required. however, the other parameters have default values and are optional. if you were to make this publically available for other people to use, they may want to use a different language or units, so making them configurable is important.
        
        self.latitude = latitude
        self.longitude = longitude
        self.units = units
        self.language = language
        self.format = format
        self.apiKey = apiKey
        
        # we know in the requests that the latitude and longitude are required, so we combine them into a single string for convenience like the api requires
        self.lat_long = f"{self.latitude},{self.longitude}"
        
        # we can now use this combined string in our parameters dictionary
        self.parameters = {
            "geocode": self.lat_long,
            "units": self.units,
            "language": self.language,
            "format": self.format,
            "apiKey": self.apiKey
        }
        
        # all of the requests will use this base URL and are suffixed with the specific endpoint
        self.base_url = "https://api.weather.com/v3/wx/forecast/"
```


this is a rudimentary implementation of the WeatherAPI class, so lets add some error and type checking to ensure the values we need are correct.

In [2]:
class WeatherAPI:
    def __init__(self, 
                 latitude: str, 
                 longitude: str,
                 units: str = "e",
                 language: str = "en-US",
                 format: str = "json",
                 apiKey: str = "71f92ea9dd2f4790b92ea9dd2f779061"
                 ):
        
        # we know lat and long need to be a concatenated string, so we cant pass in floats (decimals). but since we are using strings, the user could pass in letters, which would be invalid. we also know that latitude runs from -90 to 90 and longitude runs from -180 to 180, so we can validate the input accordingly.
        
        if not (-90 <= float(latitude) <= 90):
            raise ValueError("Latitude must be between -90 and 90.")
        if not (-180 <= float(longitude) <= 180):
            raise ValueError("Longitude must be between -180 and 180.")
        
        # the above validation ensures that the latitude and longitude are within the valid range. if the user passes in a string that cannot be converted to a float, a ValueError will be raised when trying to convert to a float, so this catches both our edge cases.
        
        # even though we set defaults, a user can still override these values and potentially try and pass in something invalid, so we should validate those as well.

        if any(param is None or param == "" for param in [units, language, format, apiKey]):
            raise ValueError("Units, language, format, or apiKey cannot be None or empty strings.")
        
        # now that weve validated, we can assign the values
        self.latitude = latitude
        self.longitude = longitude
        self.units = units
        self.language = language
        self.format = format
        self.apiKey = apiKey
        
        self.lat_long = f"{self.latitude},{self.longitude}"
        
        self.parameters = {
            "geocode": self.lat_long,
            "units": self.units,
            "language": self.language,
            "format": self.format,
            "apiKey": self.apiKey
        }
        
        self.base_url = "https://api.weather.com/v3/wx/forecast/"

In [4]:
# just to test our implementation, lets create an instance of the WeatherAPI class

WeatherAPI(latitude="37.7749", longitude="-122.4194")

In [5]:
# we successfully created an instance of the WeatherAPI class, but the __repr__ method (which provides a string representation of the object) has not been defined yet. so lets do that

class WeatherAPI:
    def __init__(self, 
                 latitude: str, 
                 longitude: str,
                 units: str = "e",
                 language: str = "en-US",
                 format: str = "json",
                 apiKey: str = "71f92ea9dd2f4790b92ea9dd2f779061"
                 ):
        
        if not (-90 <= float(latitude) <= 90):
            raise ValueError("Latitude must be between -90 and 90.")
        if not (-180 <= float(longitude) <= 180):
            raise ValueError("Longitude must be between -180 and 180.")
        
        if any(param is None or param == "" for param in [units, language, format, apiKey]):
            raise ValueError("Units, language, format, or apiKey cannot be None or empty strings.")
        
        self.latitude = latitude
        self.longitude = longitude
        self.units = units
        self.language = language
        self.format = format
        self.apiKey = apiKey
        
        self.lat_long = f"{self.latitude},{self.longitude}"
        
        self.parameters = {
            "geocode": self.lat_long,
            "units": self.units,
            "language": self.language,
            "format": self.format,
            "apiKey": self.apiKey
        }
        
        self.base_url = "https://api.weather.com/v3/wx/forecast/"

    def __repr__(self):
        return f"WeatherAPI(latitude={self.latitude}, longitude={self.longitude}, units={self.units}, language={self.language}, format={self.format}, apiKey={self.apiKey})"
    
# now when we create an instance of the WeatherAPI class, the __repr__ method will provide a string representation of the object

WeatherAPI(latitude="37.7749", longitude="-122.4194")

WeatherAPI(latitude=37.7749, longitude=-122.4194, units=e, language=en-US, format=json, apiKey=71f92ea9dd2f4790b92ea9dd2f779061)

In [ ]:
# from here, we could just start adding methods to interact with the weather API, such as fetching the current weather, forecast, etc. but there are several reusable components we can add first, such as a method to build the request URL and a method to make the API call, since these will be used with every single API call we make
import requests

class WeatherAPI:
    def __init__(self, 
                 latitude: str, 
                 longitude: str,
                 units: str = "e",
                 language: str = "en-US",
                 format: str = "json",
                 apiKey: str = "71f92ea9dd2f4790b92ea9dd2f779061"
                 ):
        
        if not (-90 <= float(latitude) <= 90):
            raise ValueError("Latitude must be between -90 and 90.")
        if not (-180 <= float(longitude) <= 180):
            raise ValueError("Longitude must be between -180 and 180.")
        
        if any(param is None or param == "" for param in [units, language, format, apiKey]):
            raise ValueError("Units, language, format, or apiKey cannot be None or empty strings.")
        
        self.latitude = latitude
        self.longitude = longitude
        self.units = units
        self.language = language
        self.format = format
        self.apiKey = apiKey
        
        self.lat_long = f"{self.latitude},{self.longitude}"
        
        self.parameters = {
            "geocode": self.lat_long,
            "units": self.units,
            "language": self.language,
            "format": self.format,
            "apiKey": self.apiKey
        }
        
        self.base_url = "https://api.weather.com/v3/wx/"

    def __repr__(self):
        return f"WeatherAPI(latitude={self.latitude}, longitude={self.longitude}, units={self.units}, language={self.language}, format={self.format}, apiKey={self.apiKey})"
    
    # methods that begin with an underscore are considered private. this means that when intellisense lists available methods, these methods will not be shown. they should only be called within the class, not outside of it
    def _build_request_url(self, endpoint: str) -> str:
        return f"{self.base_url}/{endpoint}"

    def _make_get_request(self, endpoint: str):
        # these are all steps that are common to making a GET request
        # we know that all the parameters are the same, so there is no need to have to redefine them in every single method
        # we also want to handle errors consistently, so we always want to raise_for_status before returning the response
        url = self._build_request_url(endpoint)
        response = requests.get(url, params=self.parameters)
        response.raise_for_status()
        return response.json()

In [14]:
# now, we can start adding our methods to interact with the API

class WeatherAPI:
    def __init__(self, 
                 latitude: str, 
                 longitude: str,
                 units: str = "e",
                 language: str = "en-US",
                 format: str = "json",
                 apiKey: str = "71f92ea9dd2f4790b92ea9dd2f779061"
                 ):
        
        if not (-90 <= float(latitude) <= 90):
            raise ValueError("Latitude must be between -90 and 90.")
        if not (-180 <= float(longitude) <= 180):
            raise ValueError("Longitude must be between -180 and 180.")
        
        if any(param is None or param == "" for param in [units, language, format, apiKey]):
            raise ValueError("Units, language, format, or apiKey cannot be None or empty strings.")
        
        self.latitude = latitude
        self.longitude = longitude
        self.units = units
        self.language = language
        self.format = format
        self.apiKey = apiKey
        
        self.lat_long = f"{self.latitude},{self.longitude}"
        
        self.parameters = {
            "geocode": self.lat_long,
            "units": self.units,
            "language": self.language,
            "format": self.format,
            "apiKey": self.apiKey
        }
        
        self.base_url = "https://api.weather.com/v3/wx"

    def __repr__(self):
        return f"WeatherAPI(latitude={self.latitude}, longitude={self.longitude}, units={self.units}, language={self.language}, format={self.format}, apiKey={self.apiKey})"
    
    def _build_request_url(self, endpoint: str) -> str:
        return f"{self.base_url}/{endpoint}"

    def _make_get_request(self, endpoint: str):
        url = self._build_request_url(endpoint)
        response = requests.get(url, params=self.parameters)
        response.raise_for_status()
        return response.json()
    
    def get_current_weather(self):
        # the url for current weather is https://api.weather.com/v3/wx/observations/current
        return self._make_get_request("observations/current")
    
    def get_2_day_hourly_forecast(self):
        # the url for 2-day hourly forecast is https://api.weather.com/v3/wx/forecast/hourly/2day
        return self._make_get_request("forecast/hourly/2day")
    
    def get_3_day_daily_forecast(self):
        # the url for 3-day forecast is https://api.weather.com/v3/wx/forecast/daily/3day
        return self._make_get_request("forecast/daily/3day")
    
    def get_7_day_daily_forecast(self):
        # the url for 7-day daily forecast is https://api.weather.com/v3/wx/forecast/daily/7day
        return self._make_get_request("forecast/daily/7day")
    
    
    def get_15_minute_forecast(self):
        # the url for 15-minute forecast is https://api.weather.com/v3/wx/forecast/fifteenminute
        return self._make_get_request("forecast/fifteenminute")
    
# now lets test it out:
weather_api = WeatherAPI(latitude="37.7749", longitude="-122.4194")
current = weather_api.get_current_weather()
if current:
    print(f'current weather success')
    
two_day = weather_api.get_2_day_hourly_forecast()
if two_day:
    print(f'2-day hourly forecast success')
    
three_day = weather_api.get_3_day_daily_forecast()
if three_day:
    print(f'3-day daily forecast success')
    
seven_day = weather_api.get_7_day_daily_forecast()
if seven_day:
    print(f'7-day daily forecast success')
    
fifteen_minute = weather_api.get_15_minute_forecast()
if fifteen_minute:
    print(f'15-minute forecast success')

current weather success
2-day hourly forecast success
3-day daily forecast success
7-day daily forecast success
15-minute forecast success


In [15]:
# run the below to dump the weather responses into JSON files so we can explore the outputs
import os
import json

samples_dir = './samples'

if not os.path.exists(samples_dir):
    os.makedirs(samples_dir)
    
# dump the weather responses into JSON files
with open(os.path.join(samples_dir, 'current_weather.json'), 'w') as f:
    json.dump(current, f, indent=4)

with open(os.path.join(samples_dir, '2_day_hourly_forecast.json'), 'w') as f:
    json.dump(two_day, f, indent=4)

with open(os.path.join(samples_dir, '3_day_daily_forecast.json'), 'w') as f:
    json.dump(three_day, f, indent=4)

with open(os.path.join(samples_dir, '7_day_daily_forecast.json'), 'w') as f:
    json.dump(seven_day, f, indent=4)

with open(os.path.join(samples_dir, '15_minute_forecast.json'), 'w') as f:
    json.dump(fifteen_minute, f, indent=4)